In [15]:
import ibis
import pandas as pd


# Setup backends
polars_backend = ibis.polars.connect()
duckdb_backend = ibis.duckdb.connect()
sqlite_backend = ibis.sqlite.connect()

# Create test data
pl_df = pd.DataFrame({'value': [10]})

# Create tables
df_polars = polars_backend.create_table('test', pl_df, overwrite=True)
df_duckdb = duckdb_backend.create_table('test', pl_df, overwrite=True)
df_sqlite = sqlite_backend.create_table('test', pl_df, overwrite=True)

# Define expressions using Deferred syntax
lit = ibis.literal(5)
col = ibis._['value']  # Deferred column reference

# Test each backend
results = []

for backend_name, table in [
    ('polars', df_polars),
    ('duckdb', df_duckdb),
    ('sqlite', df_sqlite),
]:
    result = {'backend': backend_name}

    # Test: col + lit (should work)
    try:
        expr = col + lit
        res = table.select(expr.name('result')).execute()
        result['col+lit'] = res.result[0]
    except Exception as e:
        result['col+lit'] = f'{type(e).__name__}'

    # Test: lit + col (fails with InputTypeError)
    try:
        expr = lit + col
        res = table.select(expr.name('result')).execute()
        result['lit+col'] = res.result[0]
    except Exception as e:
        result['lit+col'] = f'{type(e).__name__}'

    results.append(result)

# Display results
df_results = pd.DataFrame(results)
print(df_results.to_markdown(index=False))

| backend   |   col+lit | lit+col        |
|:----------|----------:|:---------------|
| polars    |        15 | InputTypeError |
| duckdb    |        15 | InputTypeError |
| sqlite    |        15 | InputTypeError |


In [11]:
df_results

,backend,col+lit,lit+col
0,polars,15,InputTypeError
1,duckdb,15,InputTypeError
2,sqlite,15,InputTypeError


In [16]:
        expr = lit + col
        res = table.select(expr.name('result')).execute()


InputTypeError: Unable to infer datatype of value _['value'] with type <class 'ibis.common.deferred.Deferred'>